In [11]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv

Configuration

In [12]:
# ============================================================
# Load environment variables
# PATH_CIV5GAME must point to the root of your Civ 5 installation.
#
# Examples:
#   Linux   : /run/media/arthur/disk/steamapps/common/Sid Meier's Civilization V
#   macOS   : /Users/arthur/Library/Application Support/Steam/steamapps/common/Sid Meier's Civilization V
#   Windows : C:/Program Files (x86)/Steam/steamapps/common/Sid Meier's Civilization V
# ============================================================
 
load_dotenv(override=True)
 
CIV5_ROOT = Path(os.environ.get("PATH_CIV5GAME", ""))
DATA_RAW  = Path("../data/raw_traits")
 
if not CIV5_ROOT or not CIV5_ROOT.exists():
    raise EnvironmentError(
        "PATH_CIV5GAME is not set or does not exist.\n"
        "Please fill in your .env file (see .env.example)."
    )
 
DATA_RAW.mkdir(parents=True, exist_ok=True)
 
print(f"Civ 5 root : {CIV5_ROOT}")
print(f"Output dir : {DATA_RAW}")


OSError: PATH_CIV5GAME is not set or does not exist.
Please fill in your .env file (see .env.example).

Define XML sources

In [4]:
# ============================================================
# XML source directories within the Civ 5 installation.
#
# Civ 5 stores leader XML files in four distinct locations:
#
#   1. Vanilla      — base game leaders (most of the roster)
#   2. Expansion    — Gods & Kings (adds ~9 leaders)
#   3. Expansion2   — Brave New World (adds ~9 leaders)
#   4. DLC_01–07    — individual civilization DLCs (6 leaders)
#   5. DLC_Deluxe   — Babylon pack (Nebuchadnezzar only)
#
# Leaders affected by DLC:
#   GenghisKhan, Harald, Isabella, Kamehameha,
#   Pachacuti, Sejong → DLC_01 to DLC_07
#   Nebuchadnezzar    → DLC_Deluxe
# ============================================================
 
SOURCES = [
    # (label, glob_pattern)
    (
        "Vanilla",
        CIV5_ROOT / "Assets" / "Gameplay" / "XML" / "Leaders"
    ),
    (
        "Expansion 1 (Gods & Kings)",
        CIV5_ROOT / "Assets" / "DLC" / "Expansion" / "Gameplay" / "XML" / "Leaders"
    ),
    (
        "Expansion 2 (Brave New World)",
        CIV5_ROOT / "Assets" / "DLC" / "Expansion2" / "Gameplay" / "XML" / "Leaders"
    ),
]
 
# Individual DLC packs (DLC_01 to DLC_07) — searched recursively
DLC_BASE = CIV5_ROOT / "Assets" / "DLC"
DLC_DIRS = [f"DLC_0{i}" for i in range(1, 8)]
 
# Babylon pack (Nebuchadnezzar)
DLC_DELUXE = CIV5_ROOT / "Assets" / "DLC" / "DLC_Deluxe" / "Gameplay" / "XML"
 
print("Sources defined.")
for label, path in SOURCES:
    status = "✓" if path.exists() else "✗ NOT FOUND"
    print(f"  [{status}] {label}: {path}")


Sources defined.
  [✗ NOT FOUND] Vanilla: Assets/Gameplay/XML/Leaders
  [✗ NOT FOUND] Expansion 1 (Gods & Kings): Assets/DLC/Expansion/Gameplay/XML/Leaders
  [✗ NOT FOUND] Expansion 2 (Brave New World): Assets/DLC/Expansion2/Gameplay/XML/Leaders


Copy function

In [5]:
def copy_xml_files(src_dir, dest_dir, label, recursive=False):
    """
    Copy all CIV5Leader*.xml files from src_dir to dest_dir.
    Skips localisation files (in /Text/ subdirectories).
 
    Args:
        src_dir   : Path — source directory
        dest_dir  : Path — destination directory (data/raw/)
        label     : str  — displayed in logs
        recursive : bool — whether to search subdirectories
 
    Returns:
        int — number of files copied
    """
    if not src_dir.exists():
        print(f"  [SKIP] {label} — directory not found: {src_dir}")
        return 0
 
    dest_dir.mkdir(parents=True, exist_ok=True)
 
    pattern = "**/*.xml" if recursive else "*.xml"
    candidates = list(src_dir.glob(pattern))
 
    copied = 0
    for f in candidates:
        if "Text" in f.parts:                  # skip localisation files
            continue
        if not f.name.startswith("CIV5Leader"):
            continue
 
        dest = dest_dir / f.name
        shutil.copy2(f, dest)
        print(f"    Copied: {f.name}")
        copied += 1
 
    if copied == 0:
        print(f"  [WARN] {label} — no CIV5Leader*.xml files found in {src_dir}")
    else:
        print(f"  [{copied} files] {label}")
 
    return copied


Run import

In [6]:
print("=" * 55)
print("Importing XML files into data/raw/")
print("=" * 55)
 
total = 0
 
# 1. Vanilla + Expansion 1 & 2
for label, src_dir in SOURCES:
    total += copy_xml_files(src_dir, DATA_RAW, label)
 
# 2. Individual DLC packs (DLC_01 to DLC_07)
print(f"\n  Searching individual DLC packs ({', '.join(DLC_DIRS)})...")
for dlc in DLC_DIRS:
    dlc_path = DLC_BASE / dlc / "Gameplay" / "XML"
    total += copy_xml_files(dlc_path, DATA_RAW, dlc, recursive=True)
 
# 3. DLC_Deluxe (Nebuchadnezzar)
neby = DLC_DELUXE / "CIV5Leader_Nebuchadnezzar.xml"
if neby.exists():
    shutil.copy2(neby, DATA_RAW / neby.name)
    print(f"  [1 file] DLC_Deluxe — {neby.name}")
    total += 1
else:
    print(f"  [SKIP] DLC_Deluxe — CIV5Leader_Nebuchadnezzar.xml not found")
 
print(f"\n{'=' * 55}")
print(f"Done. {total} XML files copied to {DATA_RAW}")


Importing XML files into data/raw/
  [SKIP] Vanilla — directory not found: Assets/Gameplay/XML/Leaders
  [SKIP] Expansion 1 (Gods & Kings) — directory not found: Assets/DLC/Expansion/Gameplay/XML/Leaders
  [SKIP] Expansion 2 (Brave New World) — directory not found: Assets/DLC/Expansion2/Gameplay/XML/Leaders

  Searching individual DLC packs (DLC_01, DLC_02, DLC_03, DLC_04, DLC_05, DLC_06, DLC_07)...
  [SKIP] DLC_01 — directory not found: Assets/DLC/DLC_01/Gameplay/XML
  [SKIP] DLC_02 — directory not found: Assets/DLC/DLC_02/Gameplay/XML
  [SKIP] DLC_03 — directory not found: Assets/DLC/DLC_03/Gameplay/XML
  [SKIP] DLC_04 — directory not found: Assets/DLC/DLC_04/Gameplay/XML
  [SKIP] DLC_05 — directory not found: Assets/DLC/DLC_05/Gameplay/XML
  [SKIP] DLC_06 — directory not found: Assets/DLC/DLC_06/Gameplay/XML
  [SKIP] DLC_07 — directory not found: Assets/DLC/DLC_07/Gameplay/XML
  [SKIP] DLC_Deluxe — CIV5Leader_Nebuchadnezzar.xml not found

Done. 0 XML files copied to ../data/raw


Verification

In [7]:
# ============================================================
# Verify that all 43 expected leaders are present.
# If some are missing, the cell prints which ones — check
# that the corresponding DLC is installed in your Civ 5.
# ============================================================
 
EXPECTED_LEADERS = [
    "AhmadalMansur", "Alexander", "Ashurbanipal", "Askia", "Attila",
    "Augustus", "Bismark", "Boudicca", "Casimir", "Catherine",
    "Darius", "Dido", "Elizabeth", "EnricoDandolo", "GajahMada",
    "Gandhi", "GenghisKhan", "Gustavus", "Harald", "HarunAlRashid",
    "Hiawatha", "Isabella", "Kamehameha", "Maria", "MariaI",
    "Montezuma", "Napoleon", "Nebuchadnezzar", "OdaNobunaga", "Pacal",
    "Pachacuti", "Pedro", "Pocatello", "Ramesses", "Ramkhamhaeng",
    "Sejong", "Selassie", "Shaka", "Suleiman", "Theodora",
    "Washington", "William", "WuZetian",
]
 
found_files = list(DATA_RAW.glob("CIV5Leader_*.xml"))
found_names = {f.stem.replace("CIV5Leader_", "") for f in found_files}
 
# Remove known non-leader files
EXCLUDES = {"Barbarian", "Tables", "Inherited_Expansion2"}
found_names = {n for n in found_names if not any(ex in n for ex in EXCLUDES)}
 
missing  = sorted(set(EXPECTED_LEADERS) - found_names)
extra    = sorted(found_names - set(EXPECTED_LEADERS))
 
print(f"XML files found in data/raw/ : {len(found_files)}")
print(f"Expected leaders             : {len(EXPECTED_LEADERS)}")
 
if missing:
    print(f"\n⚠ Missing ({len(missing)}) — check DLC installation:")
    for name in missing:
        print(f"    {name}")
else:
    print("\n✓ All 43 leaders present.")
 
if extra:
    print(f"\nExtra files (not in expected list — ignored by notebook 01):")
    for name in extra:
        print(f"    {name}")


XML files found in data/raw/ : 0
Expected leaders             : 43

⚠ Missing (43) — check DLC installation:
    AhmadalMansur
    Alexander
    Ashurbanipal
    Askia
    Attila
    Augustus
    Bismark
    Boudicca
    Casimir
    Catherine
    Darius
    Dido
    Elizabeth
    EnricoDandolo
    GajahMada
    Gandhi
    GenghisKhan
    Gustavus
    Harald
    HarunAlRashid
    Hiawatha
    Isabella
    Kamehameha
    Maria
    MariaI
    Montezuma
    Napoleon
    Nebuchadnezzar
    OdaNobunaga
    Pacal
    Pachacuti
    Pedro
    Pocatello
    Ramesses
    Ramkhamhaeng
    Sejong
    Selassie
    Shaka
    Suleiman
    Theodora
    Washington
    William
    WuZetian
